In [ ]:
import sys
sys.path.append("/public/home/guogjgroup/ggj/JiaqiLi/NvTK/")
print(sys.path)

In [ ]:
import h5py, os, argparse, logging, time

import numpy as np
import pandas as pd

import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from NvTK import Trainer
from NvTK.Model.Publications import DeepSEA

from NvTK.Evaluator import calculate_roc, calculate_pr
from NvTK.Evaluator import show_auc_curve, show_pr_curve

from NvTK.Explainer import get_activate_W, meme_generate, save_activate_seqlets
from NvTK.Explainer import seq_logo, plot_seq_logo

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def plot_tracks(tracks, interval, height=1.5, save=False):
    fig, axes = plt.subplots(len(tracks), 1, figsize=(20, height * len(tracks)), sharex=True)
    for ax, (title, y) in zip(axes, tracks.items()):
        ax.fill_between(np.linspace(interval.get("start"), interval.get("end"), num=len(y)), y)
        ax.set_title(title)
        sns.despine(top=True, right=True, bottom=True)
    ax.set_xlabel(str(interval))
    plt.tight_layout()
    plt.savefig(save) if save else None

# sns.heatmap(y_train_regulatory[:,:,0], cmap='Greys')


In [ ]:
tasks_use = ['P53_1', 'P53_10', 'P53_11', 'P53_12', 'P53_13', 'P53_14',
           'P53_15', 'P53_16', 'P53_17', 'P53_18', 'P53_19', 'P53_2',
           'P53_20', 'P53_22', 'P53_24', 'P53_25', 'P53_3', 'P53_4', 'P53_5',
           'P53_6', 'P53_7', 'P53_8', 'P53_9', 'WT_1', 'WT_10', 'WT_11',
           'WT_12', 'WT_13', 'WT_15', 'WT_2', 'WT_20', 'WT_21', 'WT_22',
           'WT_23', 'WT_24', 'WT_25', 'WT_3', 'WT_4', 'WT_5', 'WT_6', 'WT_7',
           'WT_8', 'WT_9']

cellanno_d = {i:j for i,j in enumerate(tasks_use)}

In [ ]:
os.makedirs("./Log", exist_ok=True)
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s %(filename)s[line:%(lineno)d] %(levelname)s %(message)s',
                    datefmt='%a, %d %b %Y %H:%M:%S',
                    filename=time.strftime('./Log/log_NvP53.%m%d.%H:%M:%S.txt'),
                    filemode='w')

# args
parser = argparse.ArgumentParser()
parser.add_argument("data")
parser.add_argument("--gpu-device", dest="device_id", default="0")
args = parser.parse_args(['../dataset/paired-20220804/Dataset.pmat.pb.202200805.h5', 
                          '--gpu-device', '0'])
logging.info(args)

In [ ]:
## change device
os.environ["CUDA_VISIBLE_DEVICES"] = args.device_id
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
import torch
from torch import nn
from NvTK import BasicModule
from NvTK.Modules.Transformer import TransformerEncoder

# define modules
KoScaner = nn.Sequential(
            nn.Conv1d(4, 8, 19, 1, 9), 
            nn.ReLU(),
            nn.MaxPool1d(20),
            )

WtScaner = nn.Sequential(
            nn.Conv1d(4, 8, 19, 1, 9), 
            nn.ReLU(),
            nn.MaxPool1d(20),
            )

class Residual(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn

    def forward(self, x, **kwargs):
        return self.fn(x, **kwargs) + x

SharedScaner = nn.Sequential(
            nn.Conv1d(4, 128, 11, 1, 5), nn.MaxPool1d(5),
            Residual(nn.Sequential(nn.Conv1d(128, 128, 9, 1, 4), nn.ReLU())),
            nn.Conv1d(128, 256, 7, 1, 3), nn.MaxPool1d(2),
            Residual(nn.Sequential(nn.Conv1d(256, 256, 7, 1, 3), nn.ReLU())),
            nn.Conv1d(256, 504, 5, 1, 2), nn.MaxPool1d(2),
            Residual(nn.Sequential(nn.Conv1d(504, 504, 3, 1, 1), nn.ReLU())),
            )


class SeqScaner(BasicModule):
    def __init__(self):
        super().__init__()
        self.KoScaner = KoScaner
        self.WtScaner = WtScaner
        self.SharedScaner = SharedScaner
        
        self.seq_pool_size = 20 # * 5
        self.SharedScaner_size = 504
        self.Scaner_size = 8
        self.seq_channel_size = 512 #1024

    def forward(self, x, t='Wt'):
        x0 = self.SharedScaner(x)
        if t == 'Ko':
            x1 = self.KoScaner(x)
        elif t == 'Wt':
            x1 = self.WtScaner(x)
        return [x0, x1]


class MultiPredictor(BasicModule):
    def __init__(self, n_size, n_tasks_ko=3, n_tasks_wt=2):
        super().__init__()
        self.linear_ko = nn.Linear(n_size, n_tasks_ko)
        self.linear_wt = nn.Linear(n_size, n_tasks_wt)
        
    def forward(self, x, t='Wt'):
        if t == 'Ko':
            x = self.linear_ko(x)
        elif t == 'Wt':
            x = self.linear_wt(x)
        return x


# define models
class NvATAC(BasicModule):
    def __init__(self, n_seqs=200, n_tasks_ko=3, n_tasks_wt=2):
        super().__init__()
        self.SeqScaner = SeqScaner()
        n_len = n_seqs // self.SeqScaner.seq_pool_size
        n_size = n_len * self.SeqScaner.seq_channel_size
        self.predictor = MultiPredictor(n_size, n_tasks_ko, n_tasks_wt)
        self.activate = nn.Sigmoid()

    def forward(self, x, t='Wt'):
        x = self.SeqScaner(x, t) # return a list
        x = torch.concat(x, dim=1)
        x = x.reshape(x.size(0), -1)
        x = self.predictor(x, t)
        x = self.activate(x)
        return x


class NvRNA(BasicModule):
    def __init__(self, n_seqs=2000, n_tasks_ko=3, n_tasks_wt=2):
        super().__init__()
        self.SeqScaner = SeqScaner()
        
        n_len = n_seqs // self.SeqScaner.seq_pool_size
        n_d = self.SeqScaner.SharedScaner_size
        self.rnn = nn.LSTM(input_size=n_d, hidden_size = n_d // 2 , 
                           num_layers=4, batch_first=True, bidirectional=True)
        
        n_size = n_len * (n_d + self.SeqScaner.Scaner_size)
        self.predictor = MultiPredictor(n_size, n_tasks_ko, n_tasks_wt)
        self.activate = nn.Sigmoid()

    def forward(self, x, t='Wt'):
        x0, x1 = self.SeqScaner(x, t) # bs, hidden, seq_len
        x0 = x0.swapaxes(1, -1) # bs, seq_len, embedding(batch_size,seq_length,input_size)
        x0, _ = self.rnn(x0) # bs, seq_len, hidden*2
        x0 = x0.swapaxes(1, -1) # bs, hidden*2, seq_len
        x = torch.concat([x0, x1], dim=1)
        x = x.reshape(x.size(0), -1)
        x = self.predictor(x, t)
        x = self.activate(x)
        return x


class NvP53(BasicModule):
    def __init__(self, n_seqs_rna=2000, n_tasks_ko_rna=3, n_tasks_wt_rna=2,
                    n_seqs_atac=200, n_tasks_ko_atac=3, n_tasks_wt_atac=2):
        super().__init__()
        self.NvRNA = NvRNA(n_seqs_rna, n_tasks_ko_rna, n_tasks_wt_rna)
        self.NvATAC = NvATAC(n_seqs_atac, n_tasks_ko_atac, n_tasks_wt_atac)
        # Sharing SeqScaner parameters
        self.NvRNA.SeqScaner = self.NvATAC.SeqScaner = SeqScaner()

    def forward(self, inp):
        x, xt = inp
        self.switch_xt(xt)
        if self.xt == 'rna':
            o1 = self.NvRNA(x, t='Wt')
            o2 = self.NvRNA(x, t='Ko')
        elif self.xt == 'atac':
            o1 = self.NvATAC(x, t='Wt')
            o2 = self.NvATAC(x, t='Ko')
        return [o1, o2, self.xt]

    def switch_xt(self, xt='atac'):
        self.xt = xt



In [ ]:
model = NvP53(10000, 121, 121, 1000, 23, 20)
model.load_state_dict(torch.load("../NvP53_AtacRna_KoWt_OdL6_20221021/Log/best_model.pth"), strict=True)

In [ ]:
model.to(device)

### prepare the seuqence 

In [ ]:
target_genes = pd.read_csv("./Result_trp53_target_support_sig_GroundTruth.csv", sep=',')
target_genes = target_genes[target_genes.p53target == "Ground_truth"]
target_genes.head()

In [ ]:
gene_info = pd.read_csv("../dataset/gtf_annotation.csv", index_col=0)
gene_info.head()

In [ ]:
target_info = target_genes.merge(gene_info, how='left', left_on='genename', right_index=True).sort_values("score", ascending=False)
print(target_info.shape)
target_info.head()

In [ ]:
target_info['middle'] = (target_info.start.astype(int) + target_info.end.astype(int)) // 2
target_info['length'] = target_info.end.astype(int) - target_info.start.astype(int)
target_info.head()

In [ ]:
target_info = target_info[target_info.length < 200_000]
target_info.shape

In [ ]:
chrom = '2'
start = 126158710 - 100_000
end = 126158710 + 100_000
seq_len = end - start

In [ ]:
import pysam

onehot_nuc = {'A':[1,0,0,0],
            'C':[0,1,0,0],
            'G':[0,0,1,0],
            'T':[0,0,0,1],
            'N':[0,0,0,0]}

def _onehot_seq(seq):
    return np.array([onehot_nuc[nuc] for nuc in str(seq).upper()])

In [ ]:
genome = pysam.FastaFile("../../Resource/STAR_Reference_Mouse/Mus_musculus.GRCm38.88.fasta")
genome

In [ ]:
# deal with boundary
chrom_size = genome.get_reference_length(chrom)
if end > chrom_size:
    print(peak[-1])
    pad = 'N' * (end - chrom_size) # pad N
    end = chrom_size
    
# fetch sequence 
seq = genome.fetch(reference=chrom, start=start, end=end)

# onehot    
seq = _onehot_seq(seq)
seq = seq.astype(np.float32)
seq.shape

In [ ]:
genome.close()

### scan the onehot sequence

In [ ]:
stride = 10
model_input_length = 1000
batch_size = 500

In [ ]:
scan_seq_strides_info = []
scan_seq_strides = []

for scan_start in range(0, seq_len, stride):
    scan_end = scan_start + model_input_length
    if scan_end > seq_len:
        break
    scan_seq_strides_info.append([start+scan_start, start+scan_end])
    
    scan_seq = seq[scan_start:scan_end, :]    
    scan_seq_strides.append(scan_seq)

scan_seq_strides_info = np.array(scan_seq_strides_info)
scan_seq_strides = np.array(scan_seq_strides)

scan_seq_strides.shape, scan_seq_strides_info.shape

In [ ]:
out = []
for i in range(0, scan_seq_strides.shape[0], batch_size):
    batch_seq = scan_seq_strides[i:i+batch_size, ]
    batch_seq = torch.from_numpy(batch_seq.swapaxes(1, -1)).to(device)
    batch_out = model.forward([batch_seq, 'atac'])
    batch_out = torch.concat(batch_out[:2], axis=-1)
    out.append(batch_out.cpu().data.numpy())
#     print(batch_out.shape)
    
out = np.vstack(out)
out.shape

### plot tracks

In [ ]:
os.makedirs("Figures", exist_ok=True)

In [ ]:
# interval = dict(chrom="chr15", start=start, end=end)
# tracks = {cellanno_d[i]+"__task"+str(i):out[:,i] for i in range(len(cellanno))}
# plot_tracks(tracks, interval, height=1.5, save="Figures/tracks_Myc.pdf")

In [ ]:
interval = dict(chrom=chrom, start=start, end=end)
tracks = {cellanno_d[i]+"_task"+str(i):out[:,i] for i in [27, 28, 29, 3, 30, 31, 32, 33, 18, 2]}
plot_tracks(tracks, interval, height=1.5, save="Figures/tracks_Dtwd1_Final.pdf")

### scan the sequence with in silico mutation

In [ ]:
stride = 1
model_input_length = 1000
middle = model_input_length // 2
batch_size = 200

In [ ]:
scan_seq_strides_info = []
scan_seq_strides = []
mutate_info = []

for scan_start in range(0, seq_len, stride):
    scan_end = scan_start + model_input_length
    if scan_end > seq_len:
        break
    scan_seq_strides_info.append([start+scan_start, start+scan_end])
    
    scan_seq = seq[scan_start:scan_end, :]    
    scan_seq_strides.append(scan_seq)
    
    ref = scan_seq[model_input_length//2:(model_input_length//2)+1]
    ref = np.where(ref==1)[1]
    for alt in range(4):
        mutate_info.append([start+scan_start+model_input_length//2, 
                            start+scan_start+ model_input_length//2 + 1, ref, alt])

scan_seq_strides_info = np.array(scan_seq_strides_info)
scan_seq_strides = np.array(scan_seq_strides)
mutate_info = np.array(mutate_info)

scan_seq_strides.shape, scan_seq_strides_info.shape, mutate_info.shape

In [ ]:
mutate_info[:,2] = np.asarray(mutate_info[:,2]).astype(int)
mutate_info

In [ ]:
out = []
for i in range(0, scan_seq_strides.shape[0], batch_size):
    batch_seq = scan_seq_strides[i:i+batch_size, ][:,None,:,:]
    batch_seq_repeat = np.tile(batch_seq, [1,4,1,1])
    batch_seq_repeat[:,:,middle:middle+1,:] = np.repeat(np.eye(4, 4)[None,:,None,:], 
                                                batch_seq_repeat.shape[0], axis=0).astype(float)
    batch_seq_repeat = batch_seq_repeat.reshape(-1, model_input_length, 4)
    
    batch_seq = torch.from_numpy(batch_seq_repeat.swapaxes(1, -1)).to(device)
    batch_out = model.forward([batch_seq, 'atac'])
    batch_out = torch.concat(batch_out[:2], axis=-1)
    out.append(batch_out.cpu().data.numpy())
    #print(batch_out.shape)
    
out = np.vstack(out)
out.shape

In [ ]:
pd.DataFrame(out, index=mutate_info).to_csv("./Out/Out_Mut_Dtwd1.csv.gz")

In [ ]:
out_all = out[:,28].reshape(-1, 4)
ref_mask = mutate_info[:,2] == mutate_info[:,3]
out_ref = out[ref_mask, 28]
out_mut = out[~ref_mask, 28].reshape(-1, 3).max(-1)

d = {0:'A', 1:'G', 2:'C', 3:'T'}
interval = dict(chrom="chr15", start=start, end=end)
tracks = {cellanno_d[i]+"_nuc_"+d[i]:out_all[:,i] for i in range(4)}
tracks["ref"] = out_ref
tracks["mut"] = out_mut
tracks["diff(Mut - Ref)"] = out_mut - out_ref
tracks["FoldChange(Mut/Ref)"] = (out_mut - out_ref) / (out_ref+1)

g = np.zeros_like(out_ref)
g[(g.shape[0]-13138)//2 : (g.shape[0]+13138)//2] = 1
tracks["Gene"] = g

plot_tracks(tracks, interval, height=1.5, save="Figures/tracks_Dtwd1_Mut.pdf")

In [ ]:
stride = 10
model_input_length = 1000
batch_size = 500

stride = 1
model_input_length = 1000
middle = model_input_length // 2
batch_size = 200

In [ ]:
import pysam

onehot_nuc = {'A':[1,0,0,0],
            'C':[0,1,0,0],
            'G':[0,0,1,0],
            'T':[0,0,0,1],
            'N':[0,0,0,0]}

def _onehot_seq(seq):
    return np.array([onehot_nuc[nuc] for nuc in str(seq).upper()])

In [ ]:
genome = pysam.FastaFile("../../Resource/STAR_Reference_Mouse/Mus_musculus.GRCm38.88.fasta")
genome

In [ ]:
for i in range(50, target_info.shape[0]):
    try:
        genename = target_info.values[i, 0]
        chrom, middle, gene_length = target_info.values[i, 8:]
        start = middle - 100_000
        end = middle + 100_000
        seq_len = end - start
        print(genename, chrom, str(start), str(end))

        # deal with boundary
        chrom_size = genome.get_reference_length(chrom)
        if end > chrom_size:
            print(peak[-1])
            pad = 'N' * (end - chrom_size) # pad N
            end = chrom_size

        # fetch sequence 
        seq = genome.fetch(reference=chrom, start=start, end=end)

        # onehot    
        seq = _onehot_seq(seq)
        seq = seq.astype(np.float32)
        print(seq.shape)

        scan_seq_strides_info = []
        scan_seq_strides = []

        for scan_start in range(0, seq_len, stride):
            scan_end = scan_start + model_input_length
            if scan_end > seq_len:
                break
            scan_seq_strides_info.append([start+scan_start, start+scan_end])

            scan_seq = seq[scan_start:scan_end, :]    
            scan_seq_strides.append(scan_seq)

        scan_seq_strides_info = np.array(scan_seq_strides_info)
        scan_seq_strides = np.array(scan_seq_strides)

        scan_seq_strides_info = []
        scan_seq_strides = []
        mutate_info = []

        for scan_start in range(0, seq_len, stride):
            scan_end = scan_start + model_input_length
            if scan_end > seq_len:
                break
            scan_seq_strides_info.append([start+scan_start, start+scan_end])

            scan_seq = seq[scan_start:scan_end, :]    
            scan_seq_strides.append(scan_seq)

            ref = scan_seq[model_input_length//2:(model_input_length//2)+1]
            ref = np.where(ref==1)[1]
            for alt in range(4):
                mutate_info.append([start+scan_start+model_input_length//2, 
                                    start+scan_start+ model_input_length//2 + 1, ref, alt])

        scan_seq_strides_info = np.array(scan_seq_strides_info)
        scan_seq_strides = np.array(scan_seq_strides)
        mutate_info = np.array(mutate_info)

        out = []
        for idx in range(0, scan_seq_strides.shape[0], batch_size):
            batch_seq = scan_seq_strides[idx:idx+batch_size, ][:,None,:,:]
            batch_seq_repeat = np.tile(batch_seq, [1,4,1,1])
            batch_seq_repeat[:,:,middle:middle+1,:] = np.repeat(np.eye(4, 4)[None,:,None,:], 
                                                        batch_seq_repeat.shape[0], axis=0).astype(float)
            batch_seq_repeat = batch_seq_repeat.reshape(-1, model_input_length, 4)

            batch_seq = torch.from_numpy(batch_seq_repeat.swapaxes(1, -1)).to(device)
            batch_out = model.forward([batch_seq, 'atac'])
            batch_out = torch.concat(batch_out[:2], axis=-1)
            out.append(batch_out.cpu().data.numpy())
            #print(batch_out.shape)

        out = np.vstack(out)
        pd.DataFrame(out, index=mutate_info).to_csv("./Out/Out_Mut_"+genename+".csv.gz")

        out_all = out[:,28].reshape(-1, 4)
        ref_mask = mutate_info[:,2] == mutate_info[:,3]
        out_ref = out[ref_mask, 28]
        out_mut = out[~ref_mask, 28].reshape(-1, 3).mean(-1)

        d = {0:'A', 1:'G', 2:'C', 3:'T'}
        interval = dict(chrom="chr15", start=start, end=end)
        tracks = {cellanno_d[i]+"_nuc_"+d[i]:out_all[:,i] for i in range(4)}
        tracks["ref"] = out_ref
        tracks["mut"] = out_mut
        tracks["diff(Mut - Ref)"] = out_mut - out_ref
        tracks["FoldChange(Mut/Ref)"] = (out_mut - out_ref) / (out_ref+1)

        g = np.zeros_like(out_ref)
        g[(g.shape[0]-gene_length)//2 : (g.shape[0]+gene_length)//2] = 1
        tracks["Gene"] = g

        plot_tracks(tracks, interval, height=1.5, save="Figures/tracks_"+genename+"_Mut.pdf")

    except:
        pass

In [ ]:
genome.close()